# 02 — Customer Intelligence: 10 Business SQL Queries

**Dataset:** 130,038 real Amazon Electronics reviews (5-core subset)
**Engine:** SQLite in-memory (loaded from CSV)
**Source:** http://jmcauley.ucsd.edu/data/amazon/

In [1]:
import pandas as pd
import sqlite3
import numpy as np

# Load into SQLite
df = pd.read_csv('../data/amazon_reviews_electronics_5core.csv')
df['reviewDate'] = pd.to_datetime(df['unixReviewTime'], unit='s')
df['reviewYear'] = df['reviewDate'].dt.year
df['reviewMonth'] = df['reviewDate'].dt.month
df['reviewLength'] = df['reviewText'].fillna('').str.len()

conn = sqlite3.connect(':memory:')
df.to_sql('reviews', conn, index=False, if_exists='replace')

print(f'Loaded {len(df):,} rows into SQLite')
print(f'Date range: {df.reviewDate.min().date()} to {df.reviewDate.max().date()}')

Loaded 130,038 rows into SQLite
Date range: 1999-11-23 to 2014-07-23


## Q1 — Product Performance Ranking

Rank products by avg rating, review volume, and helpfulness (minimum 10 reviews).

In [2]:
query1 = """
SELECT 
    asin AS product_id,
    COUNT(*) AS review_volume,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    SUM(helpful_total) AS total_votes
FROM reviews
GROUP BY asin
HAVING review_volume >= 10
ORDER BY avg_rating DESC, review_volume DESC
LIMIT 15
"""
pd.read_sql(query1, conn)

,product_id,review_volume,avg_rating,avg_helpfulness,total_votes
0,B001W26TIW,18,5.0,0.67,3
1,B003RQBKLC,16,5.0,0.86,90
2,B0061RJSWC,15,5.0,1.00,2
3,B0002BEQJ8,13,5.0,0.94,18
4,B004GW25WY,13,5.0,1.00,7
5,B005LJQNQU,13,5.0,NaN,0
6,B00B1862X4,13,5.0,0.00,1
7,B000F0ELOG,12,5.0,0.92,8
8,B000NONHYY,12,5.0,0.92,18
9,B000P1O73A,12,5.0,0.50,2


## Q2 — Rating Degradation Over Time

Are products losing trust? Track average rating by product age cohort.

In [3]:
query2 = """
WITH product_first_review AS (
    SELECT asin, MIN(reviewDate) AS first_review_date
    FROM reviews
    GROUP BY asin
),
product_age AS (
    SELECT 
        r.asin,
        r.overall,
        r.reviewDate,
        p.first_review_date,
        CAST((julianday(r.reviewDate) - julianday(p.first_review_date)) / 30 AS INTEGER) AS months_since_first
    FROM reviews r
    JOIN product_first_review p ON r.asin = p.asin
)
SELECT 
    months_since_first AS product_age_months,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating
FROM product_age
WHERE months_since_first BETWEEN 0 AND 24
GROUP BY months_since_first
ORDER BY months_since_first
"""
pd.read_sql(query2, conn)

,product_age_months,review_count,avg_rating
0,0,44511,4.18
1,1,4282,4.19
2,2,3893,4.22
3,3,3712,4.20
4,4,3524,4.25
5,5,3421,4.20
6,6,3151,4.26
7,7,3081,4.21
8,8,3098,4.24
9,9,2768,4.23


## Q3 — Review Velocity Analysis

Reviews per month, aggregated.

In [4]:
query3 = """
SELECT 
    CAST(strftime('%Y', reviewDate) AS INTEGER) AS year,
    CAST(strftime('%m', reviewDate) AS INTEGER) AS month,
    COUNT(*) AS reviews_per_month,
    ROUND(AVG(overall), 2) AS avg_rating
FROM reviews
GROUP BY year, month
ORDER BY year, month
"""
velocity = pd.read_sql(query3, conn)
print(f'Months covered: {len(velocity)}')
print(f'Avg monthly volume: {velocity.reviews_per_month.mean():.0f}')
print(f'Peak month volume: {velocity.reviews_per_month.max():,}')
velocity.tail(12)

Months covered: 177
Avg monthly volume: 735
Peak month volume: 5,142


,year,month,reviews_per_month,avg_rating
165,2013,8,3575,4.29
166,2013,9,3072,4.28
167,2013,10,3399,4.25
168,2013,11,3856,4.26
169,2013,12,5053,4.23
170,2014,1,5142,4.29
171,2014,2,4026,4.30
172,2014,3,4190,4.31
173,2014,4,3649,4.25
174,2014,5,3434,4.28


## Q4 — Helpfulness Scoring: What Makes a Review Helpful?

In [5]:
query4 = """
SELECT 
    CASE 
        WHEN reviewLength < 100 THEN 'Short (<100)'
        WHEN reviewLength < 500 THEN 'Medium (100-500)'
        ELSE 'Long (500+)'
    END AS length_tier,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    ROUND(AVG(helpful_total), 1) AS avg_votes_received
FROM reviews
GROUP BY length_tier
ORDER BY avg_helpfulness DESC
"""
pd.read_sql(query4, conn)

,length_tier,review_count,avg_rating,avg_helpfulness,avg_votes_received
0,Long (500+),47811,4.04,0.80,8.1
1,Medium (100-500),79153,4.33,0.70,1.1
2,Short (<100),3074,4.42,0.56,1.9


## Q5 — Sentiment Shift Detection (Year-over-Year Rating Changes)

In [6]:
query5 = """
SELECT 
    CAST(strftime('%Y', reviewDate) AS INTEGER) AS year,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    ROUND(AVG(reviewLength), 0) AS avg_review_length
FROM reviews
WHERE CAST(strftime('%Y', reviewDate) AS INTEGER) BETWEEN 2005 AND 2014
GROUP BY year
ORDER BY year
"""
pd.read_sql(query5, conn)

,year,review_count,avg_rating,avg_helpfulness,avg_review_length
0,2005,711,3.91,0.82,1121.0
1,2006,1107,3.91,0.83,1115.0
2,2007,2656,4.13,0.80,786.0
3,2008,3871,4.17,0.81,865.0
4,2009,5350,4.14,0.80,929.0
5,2010,7894,4.13,0.79,896.0
6,2011,13411,4.14,0.77,807.0
7,2012,21862,4.20,0.74,692.0
8,2013,45757,4.28,0.70,514.0
9,2014,26366,4.28,0.71,473.0


## Q6 — Product Lifecycle: Early Reviews vs. Mature Reviews

In [7]:
query6 = """
WITH ranked_reviews AS (
    SELECT 
        asin,
        overall,
        reviewDate,
        helpful_total,
        helpful_upvotes,
        ROW_NUMBER() OVER (PARTITION BY asin ORDER BY reviewDate) AS review_seq,
        COUNT(*) OVER (PARTITION BY asin) AS total_reviews
    FROM reviews
)
SELECT 
    CASE 
        WHEN review_seq <= 3 THEN 'Early (1st-3rd)'
        WHEN review_seq <= 10 THEN 'Growth (4th-10th)'
        ELSE 'Mature (11th+)'
    END AS lifecycle_stage,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness
FROM ranked_reviews
WHERE total_reviews >= 5
GROUP BY lifecycle_stage
ORDER BY 
    CASE lifecycle_stage
        WHEN 'Early (1st-3rd)' THEN 1
        WHEN 'Growth (4th-10th)' THEN 2
        ELSE 3
    END
"""
pd.read_sql(query6, conn)

,lifecycle_stage,review_count,avg_rating,avg_helpfulness
0,Early (1st-3rd),18138,4.25,0.75
1,Growth (4th-10th),28248,4.23,0.74
2,Mature (11th+),26457,4.37,0.70


## Q7 — Review Length Correlation with Helpfulness and Rating

In [8]:
query7 = """
SELECT 
    ROUND(overall) AS star_rating,
    ROUND(AVG(reviewLength), 0) AS avg_length,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    ROUND(AVG(helpful_total), 1) AS avg_total_votes
FROM reviews
WHERE helpful_total > 0
GROUP BY ROUND(overall)
ORDER BY star_rating
"""
pd.read_sql(query7, conn)

,star_rating,avg_length,avg_helpfulness,avg_total_votes
0,1.0,754.0,0.57,9.4
1,2.0,961.0,0.60,8.8
2,3.0,1043.0,0.64,10.9
3,4.0,1142.0,0.78,8.9
4,5.0,867.0,0.81,8.0


## Q8 — Seasonal Patterns in Review Volume

Black Friday (Nov), Prime Day (July proxy), Holiday (Dec).

In [9]:
query8 = """
SELECT 
    CAST(strftime('%m', reviewDate) AS INTEGER) AS month,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating
FROM reviews
GROUP BY month
ORDER BY month
"""
seasonal = pd.read_sql(query8, conn)
print(seasonal.to_string(index=False))

 month  review_count  avg_rating
     1         15133        4.26
     2         11817        4.27
     3         12210        4.24
     4         11165        4.22
     5         10963        4.22
     6         10756        4.21
     7         10176        4.20
     8          8061        4.18
     9          7753        4.19
    10          8089        4.20
    11          9643        4.20
    12         14272        4.22


## Q9 — Customer Engagement Tiers (Power Reviewers vs. Casual)

In [10]:
query9 = """
SELECT 
    CASE 
        WHEN review_count >= 50 THEN 'Power (50+)'
        WHEN review_count >= 10 THEN 'Active (10-49)'
        WHEN review_count >= 3 THEN 'Casual (3-9)'
        ELSE 'One-off (1-2)'
    END AS engagement_tier,
    COUNT(*) AS reviewer_count,
    ROUND(AVG(review_count), 1) AS avg_reviews_per_reviewer,
    ROUND(AVG(avg_rating), 2) AS avg_rating_given,
    ROUND(SUM(review_count) * 100.0 / (SELECT COUNT(*) FROM reviews), 1) AS pct_of_total_reviews
FROM (
    SELECT 
        reviewerID,
        COUNT(*) AS review_count,
        AVG(overall) AS avg_rating
    FROM reviews
    GROUP BY reviewerID
)
GROUP BY engagement_tier
ORDER BY avg_reviews_per_reviewer DESC
"""
pd.read_sql(query9, conn)

,engagement_tier,reviewer_count,avg_reviews_per_reviewer,avg_rating_given,pct_of_total_reviews
0,Active (10-49),183,13.3,4.29,1.9
1,Casual (3-9),7649,3.7,4.24,21.5
2,One-off (1-2),80485,1.2,4.21,76.6


## Q10 — Churn Signal: Products with Declining Review Ratings

In [11]:
query10 = """
WITH product_trend AS (
    SELECT 
        asin,
        CAST(strftime('%Y', reviewDate) AS INTEGER) AS year,
        AVG(overall) AS year_avg_rating,
        COUNT(*) AS year_review_count
    FROM reviews
    GROUP BY asin, year
    HAVING year_review_count >= 5
),
yoy_change AS (
    SELECT 
        a.asin,
        a.year AS current_year,
        a.year_avg_rating AS current_rating,
        b.year_avg_rating AS prior_rating,
        a.year_avg_rating - b.year_avg_rating AS rating_change
    FROM product_trend a
    JOIN product_trend b ON a.asin = b.asin AND a.year = b.year + 1
    WHERE b.year_avg_rating > 0
)
SELECT 
    current_year AS year,
    COUNT(*) AS products_with_data,
    ROUND(AVG(rating_change), 2) AS avg_rating_change,
    COUNT(CASE WHEN rating_change < -0.5 THEN 1 END) AS sharp_decliners,
    COUNT(CASE WHEN rating_change > 0.5 THEN 1 END) AS improvers
FROM yoy_change
GROUP BY current_year
ORDER BY current_year
"""
pd.read_sql(query10, conn)

,year,products_with_data,avg_rating_change,sharp_decliners,improvers
0,2005,1,0.20,0,0
1,2007,3,0.11,0,0
2,2008,10,0.20,1,1
3,2009,26,-0.01,4,3
4,2010,40,-0.12,8,4
5,2011,66,-0.13,18,8
6,2012,180,-0.08,33,25
7,2013,458,0.03,61,70
8,2014,593,-0.01,85,79
